## 18. Macro Stress Scenarios

In [ ]:
# ============================================================
# CELL 1 — Document the limitation, set the assumed hazard ratio
# ============================================================

# The dataset's own fitted unemployment hazard ratio (0.96) was found to be
# unreliable due to an identification problem: origination vintage and
# unemployment rate are correlated at r=-0.755 in this sample, meaning there
# isn't enough independent variation to separate "calendar time drift" from
# "macro effect." Diagnostics (crosstab of issue_year x unemployment bucket)
# confirmed near-zero overlap between vintages across unemployment levels.
#
# Rather than use the dataset's own unreliable coefficient, this stress test
# uses an assumed hazard ratio consistent with published consumer-credit
# research (Gerardi et al., Federal Reserve Bank of Atlanta — unemployment
# raises default probability by 5-13 percentage points in mortgage data).

ASSUMED_UNRATE_HR = 1.12   # midpoint of a defensible 1.10-1.15 literature-based range

print(f"Using assumed unemployment hazard ratio: {ASSUMED_UNRATE_HR}")
print(f"(Dataset's own fitted value was {unrate_hr:.4f} — rejected as unreliable; see writeup.)")

unrate_hr = ASSUMED_UNRATE_HR  # overwrite so downstream stress-test cells use this

In [ ]:
# ============================================================
# CELL 2 — Define the three stress scenarios
# ============================================================
scenarios = {
    'Baseline':         {'unemployment_delta': 0.0},
    'Adverse':          {'unemployment_delta': 3.0},
    'Severely Adverse':  {'unemployment_delta': 6.0},  # roughly 2008-magnitude shock
}

for name, shock in scenarios.items():
    print(f"{name:<18} unemployment +{shock['unemployment_delta']:.1f} pts")

In [ ]:
# ============================================================
# CELL 3 — Apply the hazard-ratio multiplier to PD, recompute portfolio EL
# ============================================================
baseline_pd = all_pd.copy()  # from Week 1's EAD + Expected Loss section

stress_results = {}
for name, shock in scenarios.items():
    multiplier = unrate_hr ** shock['unemployment_delta']
    stressed_pd = (baseline_pd * multiplier).clip(0, 1)
    stressed_el = stressed_pd * all_lgd * all_ead
    stress_results[name] = {
        'pd_multiplier': multiplier,
        'mean_pd': stressed_pd.mean(),
        'total_el': stressed_el.sum(),
        'el_rate': stressed_el.sum() / all_loan_amnt.sum(),
    }

stress_summary = pd.DataFrame(stress_results).T
stress_summary['el_increase_vs_baseline'] = (
    stress_summary['total_el'] / stress_summary.loc['Baseline', 'total_el'] - 1
)
print(stress_summary.round(4))

In [ ]:
# ============================================================
# CELL 4 — Plot EL rate under each scenario
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
stress_summary['el_rate'].plot(kind='bar', color=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax)
ax.set_title('Portfolio EL Rate Under Stress Scenarios\n(assumed hazard ratio = 1.12/pt, literature-based)')
ax.set_ylabel('EL Rate')
for i, v in enumerate(stress_summary['el_rate']):
    ax.text(i, v + 0.002, f'{v:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Transition Matrix by Economic Regime

In [ ]:
# ============================================================
# CELL 5 — Transition matrix split: expansion vs. recession regime
# (kept from the original plan — separate exercise, not affected by the
# hazard-ratio issue above, but carries its own small-sample caveat)
# ============================================================
RECESSION_START = pd.Timestamp('2007-12-01')
RECESSION_END   = pd.Timestamp('2009-06-01')

survival_df_regime = survival_df.copy()
survival_df_regime['issue_d'] = issue_dates.reindex(survival_df_regime.index)
survival_df_regime['regime'] = np.where(
    (survival_df_regime['issue_d'] >= RECESSION_START) & (survival_df_regime['issue_d'] <= RECESSION_END),
    'recession', 'expansion'
)

print(survival_df_regime['regime'].value_counts())

In [ ]:
# ============================================================
# CELL 6 — Build and compare the two regime-specific transition matrices
# ============================================================
def build_transition_matrix_for_subset(subset_df, n_states=6):
    sample_n = min(20000, len(subset_df))
    sample = subset_df.sample(n=sample_n, random_state=42) if sample_n > 0 else subset_df
    trans = build_monthly_transitions(sample)  # reuse Week 1 function
    counts = np.zeros((n_states, n_states))
    for _, row in trans.iterrows():
        counts[int(row['from_state'])][int(row['to_state'])] += 1
    counts[1] = [60, 15, 20, 0, 0, 5]
    counts[2] = [30, 10, 15, 35, 5, 5]
    counts[3] = [10, 5, 10, 20, 50, 5]
    counts[4][4] = 1
    counts[5][5] = 1
    tm = np.zeros((n_states, n_states))
    for i in range(n_states):
        rs = counts[i].sum()
        if rs > 0:
            tm[i] = counts[i] / rs
    return tm, sample_n

tm_expansion, n_exp = build_transition_matrix_for_subset(
    survival_df_regime[survival_df_regime['regime'] == 'expansion'])
tm_recession, n_rec = build_transition_matrix_for_subset(
    survival_df_regime[survival_df_regime['regime'] == 'recession'])

print(f"Expansion matrix built on {n_exp:,} loans")
print(f"Recession matrix built on {n_rec:,} loans  <-- small sample, treat as indicative only")

In [ ]:
# ============================================================
# CELL — Honest regime comparison: report only the data-driven row,
# label the rest as fixed industry assumptions (not data-derived)
# ============================================================

print("=" * 70)
print("TRANSITION MATRIX BY REGIME — WHAT'S ACTUALLY DATA-DRIVEN")
print("=" * 70)
print("""
Only the 'Current' row of this transition matrix is estimated from real
loan data. Rows for 1-30 DPD, 31-60 DPD, and 61-90 DPD are fixed industry
roll-rate assumptions (carried over from the Week 1 baseline matrix) because
the dataset only records a loan's FINAL status, not its month-by-month
delinquency path — there isn't enough data to estimate these transitions
separately by regime. Using fixed assumptions for both regimes means any
comparison on those rows is NOT a real recession-vs-expansion finding;
it would just reproduce the same numbers by construction.

The 'Current' row IS estimated separately per regime below, and is the
only honest comparison point in this matrix.
""")

current_row_compare = pd.DataFrame({
    'Expansion': tm_expansion[0],
    'Recession': tm_recession[0],
}, index=state_names)
current_row_compare['abs_diff'] = (current_row_compare['Recession'] - current_row_compare['Expansion']).round(4)

print(f"--- 'Current' row transition probabilities (n_expansion={n_exp:,}, n_recession={n_rec:,}) ---")
print(current_row_compare.round(4))

print(f"\nNote: n_recession is small ({n_rec:,} loans, all from the narrow 2007-2009 "
      f"window) relative to n_expansion ({n_exp:,} loans). Treat any differences "
      f"here as directionally suggestive, not statistically robust — same caveat "
      f"as flagged for the unemployment hazard ratio finding earlier in Section 17.")

In [ ]:
# ============================================================
# CELL — Visualize ONLY the comparable row, not the full misleading heatmaps
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(state_names))
width = 0.35

ax.bar(x - width/2, tm_expansion[0], width, label=f'Expansion (n={n_exp:,})', color='#2ecc71')
ax.bar(x + width/2, tm_recession[0], width, label=f'Recession (n={n_rec:,})', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(state_names, rotation=45, ha='right')
ax.set_ylabel('Transition Probability')
ax.set_title("'Current' Loan Transitions by Regime\n(the only data-driven row in this matrix)")
ax.legend()
plt.tight_layout()
plt.show()